In [1]:
!pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Employee.csv to Employee.csv


In [4]:
df = spark.read.csv(
    "Employee.csv",
    header=True,
    inferSchema=True
)

In [5]:
df.show(5)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|         No|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|         No|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|         No|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|         No|                        5|         1|
|  Masters|       2017|     Pune|          3| 24|  Male|        Yes|                        2|         1|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
only showing top 5 rows


In [6]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- JoiningYear: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



In [7]:
df.count()

4653

In [8]:
df.select("Education", "Age", "City").show(10)

+---------+---+---------+
|Education|Age|     City|
+---------+---+---------+
|Bachelors| 34|Bangalore|
|Bachelors| 28|     Pune|
|Bachelors| 38|New Delhi|
|  Masters| 27|Bangalore|
|  Masters| 24|     Pune|
|Bachelors| 22|Bangalore|
|Bachelors| 38|New Delhi|
|Bachelors| 34|Bangalore|
|Bachelors| 23|     Pune|
|  Masters| 37|New Delhi|
+---------+---+---------+
only showing top 10 rows


In [9]:
from pyspark.sql.functions import col

df.filter((col("Age") > 30) & (col("PaymentTier") == 3)).show(10)

+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-----------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|         No|                        0|         0|
|Bachelors|       2014|New Delhi|          3| 38|Female|         No|                        2|         0|
|Bachelors|       2015|New Delhi|          3| 38|  Male|         No|                        0|         0|
|Bachelors|       2016|Bangalore|          3| 34|Female|         No|                        2|         1|
|Bachelors|       2016|     Pune|          3| 34|  Male|         No|                        3|         0|
|Bachelors|       2018|     Pune|          3| 32|  Male|        Yes|                        5|         1|
|Bachelors|       2016|Bangalore|          3| 

In [10]:
df = df.withColumnRenamed("JoiningYear", "YearOfJoining")

In [11]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- YearOfJoining: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



In [12]:
df = df.withColumn(
    "PaymentTier",
    col("PaymentTier").cast("double")
)

In [13]:
df.printSchema()

root
 |-- Education: string (nullable = true)
 |-- YearOfJoining: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- PaymentTier: double (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EverBenched: string (nullable = true)
 |-- ExperienceInCurrentDomain: integer (nullable = true)
 |-- LeaveOrNot: integer (nullable = true)



In [14]:
from pyspark.sql.functions import when

df = df.withColumn(
    "ExperienceLevel",
    when(col("ExperienceInCurrentDomain") >= 4, "Experienced")
    .otherwise("Beginner")
)

In [15]:
df.select(
    "ExperienceInCurrentDomain",
    "ExperienceLevel"
).show(10)

+-------------------------+---------------+
|ExperienceInCurrentDomain|ExperienceLevel|
+-------------------------+---------------+
|                        0|       Beginner|
|                        3|       Beginner|
|                        2|       Beginner|
|                        5|    Experienced|
|                        2|       Beginner|
|                        0|       Beginner|
|                        0|       Beginner|
|                        2|       Beginner|
|                        1|       Beginner|
|                        2|       Beginner|
+-------------------------+---------------+
only showing top 10 rows


In [16]:
from pyspark.sql.functions import sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+---------+-------------+----+-----------+---+------+-----------+-------------------------+----------+---------------+
|Education|YearOfJoining|City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|ExperienceLevel|
+---------+-------------+----+-----------+---+------+-----------+-------------------------+----------+---------------+
|        0|            0|   0|          0|  0|     0|          0|                        0|         0|              0|
+---------+-------------+----+-----------+---+------+-----------+-------------------------+----------+---------------+



In [17]:
df = df.fillna({
    "Age": 0,
    "ExperienceInCurrentDomain": 0
})

In [18]:
filtered_df = df.filter(col("Age") > 30)

In [19]:
filtered_df.show()

+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Education|YearOfJoining|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|ExperienceLevel|
+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Bachelors|         2017|Bangalore|        3.0| 34|  Male|         No|                        0|         0|       Beginner|
|Bachelors|         2014|New Delhi|        3.0| 38|Female|         No|                        2|         0|       Beginner|
|Bachelors|         2015|New Delhi|        3.0| 38|  Male|         No|                        0|         0|       Beginner|
|Bachelors|         2016|Bangalore|        3.0| 34|Female|         No|                        2|         1|       Beginner|
|  Masters|         2017|New Delhi|        2.0| 37|  Male|         No|                        2|         0|       Beginner|
|Bachelo

In [20]:
df.groupBy("City").count().show()

+---------+-----+
|     City|count|
+---------+-----+
|Bangalore| 2228|
|     Pune| 1268|
|New Delhi| 1157|
+---------+-----+



In [21]:
df.groupBy("Gender").count().show()

+------+-----+
|Gender|count|
+------+-----+
|Female| 1875|
|  Male| 2778|
+------+-----+



In [22]:
df.write \
.mode("overwrite") \
.option("header", True) \
.csv("Processed_Employee_CSV")

In [23]:
df.write \
.mode("overwrite") \
.parquet("Processed_Employee_Parquet")

In [24]:
parquet_df = spark.read.parquet("Processed_Employee_Parquet")

parquet_df.show(5)

+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Education|YearOfJoining|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|ExperienceLevel|
+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Bachelors|         2017|Bangalore|        3.0| 34|  Male|         No|                        0|         0|       Beginner|
|Bachelors|         2013|     Pune|        1.0| 28|Female|         No|                        3|         1|       Beginner|
|Bachelors|         2014|New Delhi|        3.0| 38|Female|         No|                        2|         0|       Beginner|
|  Masters|         2016|Bangalore|        3.0| 27|  Male|         No|                        5|         1|    Experienced|
|  Masters|         2017|     Pune|        3.0| 24|  Male|        Yes|                        2|         1|       Beginner|
+-------

In [25]:
parquet_df.filter(col("Age") > 30).show()

+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Education|YearOfJoining|     City|PaymentTier|Age|Gender|EverBenched|ExperienceInCurrentDomain|LeaveOrNot|ExperienceLevel|
+---------+-------------+---------+-----------+---+------+-----------+-------------------------+----------+---------------+
|Bachelors|         2017|Bangalore|        3.0| 34|  Male|         No|                        0|         0|       Beginner|
|Bachelors|         2014|New Delhi|        3.0| 38|Female|         No|                        2|         0|       Beginner|
|Bachelors|         2015|New Delhi|        3.0| 38|  Male|         No|                        0|         0|       Beginner|
|Bachelors|         2016|Bangalore|        3.0| 34|Female|         No|                        2|         1|       Beginner|
|  Masters|         2017|New Delhi|        2.0| 37|  Male|         No|                        2|         0|       Beginner|
|Bachelo

In [26]:
parquet_df.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: parquet, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
Education: string, YearOfJoining: int, City: string, PaymentTier: double, Age: int, Gender: string, EverBenched: string, ExperienceInCurrentDomain: int, LeaveOrNot: int, ExperienceLevel: string
Relation [Education#340,YearOfJoining#341,City#342,PaymentTier#343,Age#344,Gender#345,EverBenched#346,ExperienceInCurrentDomain#347,LeaveOrNot#348,ExperienceLevel#349] parquet

== Optimized Logical Plan ==
Relation [Education#340,YearOfJoining#341,City#342,PaymentTier#343,Age#344,Gender#345,EverBenched#346,ExperienceInCurrentDomain#347,LeaveOrNot#348,ExperienceLevel#349] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [Education#340,YearOfJoining#341,City#342,PaymentTier#343,Age#344,Gender#345,EverBenched#346,ExperienceInCurrentDomain#347,LeaveOrNot#348,ExperienceLevel#349] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileInd

In [27]:
from google.colab import files
import shutil

# Zip CSV output
shutil.make_archive("Processed_Employee_CSV", "zip", "Processed_Employee_CSV")

# Zip Parquet output
shutil.make_archive("Processed_Employee_Parquet", "zip", "Processed_Employee_Parquet")

# Download
files.download("Processed_Employee_CSV.zip")
files.download("Processed_Employee_Parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Performance Insights

1. Apache Spark uses lazy evaluation, where transformations are not executed immediately. Instead, Spark creates a Directed Acyclic Graph (DAG) and optimizes the execution plan before performing any action.

2. Transformations such as select(), filter(), and withColumn() are lazy operations, whereas actions like show(), count(), and write() trigger execution.

3. Wide transformations such as groupBy() require a shuffle operation, where data is redistributed across partitions. This increases execution time compared to narrow transformations.

4. Parquet is a columnar storage format that provides better performance than CSV because it supports compression, stores schema information, and enables Predicate Pushdown.

5. Predicate Pushdown allows Spark to read only the required rows and columns from Parquet files, reducing disk I/O and improving query performance.

6. Using show() instead of collect() is recommended for large datasets because collect() transfers the entire dataset to the Driver, which can cause memory issues.

7. The data pipeline implemented in this assignment followed the sequence:
Read CSV → Transform → Filter → Add Columns → Save as CSV → Save as Parquet.


## Apache Spark Architecture

Driver:
- Creates the SparkSession.
- Converts user code into execution stages.
- Schedules tasks and collects results.

Cluster Manager:
- Allocates resources to Spark applications.
- Examples include Standalone, YARN, and Kubernetes.

Executors:
- Execute tasks assigned by the Driver.
- Store data in memory and perform computations.

Execution Modes:
- Local Mode: Runs on a single machine, suitable for development and testing.
- Cluster Mode: Runs on multiple machines, suitable for large-scale data processing.

## Lazy Evaluation

Spark uses Lazy Evaluation, meaning transformations are not executed immediately. Instead, Spark records the sequence of operations and creates a Directed Acyclic Graph (DAG). When an action such as show(), count(), or write() is called, Spark optimizes the execution plan and executes all pending transformations efficiently.

Examples of Transformations:
- select()
- filter()
- withColumn()
- groupBy()

Examples of Actions:
- show()
- count()
- collect()
- write()

# Week 6 - Spark Assignment (Updated)
This notebook covers all 15 assignment questions.

## Q1: Roles of Driver, Cluster Manager, and Executor
- **Driver:** Creates SparkSession, builds DAG, schedules tasks.
- **Cluster Manager:** Allocates resources and launches executors.
- **Executors:** Execute tasks and store partitions in memory.

## Q2: Lazy Evaluation
Spark delays execution until an action is called, allowing it to optimize the DAG and reduce unnecessary computation.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Week6SparkAssignment").getOrCreate()

df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
df.show(5)


## Q3
Read CSV with header=True and inferSchema=True (shown above).

## Q4: CSV vs Parquet
- CSV: Row-based, larger, slower.
- Parquet: Columnar, compressed, faster analytics and predicate pushdown.

In [ ]:
# Q5
df.filter(col("category")=="Electronics").select("product_id","price").show()


In [ ]:
# Q6
df = df.withColumnRenamed("old_name","new_name")
df = df.withColumn("price", col("price").cast("double"))
df.printSchema()


## Q7: DAG / Lineage
Spark records all transformations in a lineage graph. If an executor fails, only lost partitions are recomputed.

In [ ]:
# Q8
df.filter((col("status")=="Completed") & (col("amount")>1000)).show()


## Q9: Predicate Pushdown
When reading Parquet, filters are pushed to the storage layer so only matching data is read into memory.

In [ ]:
# Q10
df = df.withColumn("final_price", col("base_price")*1.18)
df.select("base_price","final_price").show()


## Q11: Transformations vs Actions
**Transformations:** select(), filter()

**Actions:** show(), collect()

Transformations are lazy; actions trigger execution.

In [ ]:
# Q12
(df.write.mode("overwrite").parquet("path/to/input"))

(spark.read.parquet("path/to/input")
      .filter(col("user_id").isNotNull())
      .write.mode("overwrite")
      .option("header",True)
      .csv("path/to/output"))


## Q13: Client Mode vs Cluster Mode
Client Mode: Driver runs on client.
Cluster Mode: Driver runs in the cluster.

In [ ]:
# Q14
df.filter((col("region")=="North") | (col("priority")=="High")).show()


## Q15: show() vs collect()
`show(5)` fetches only a few rows. `collect()` brings the entire dataset to the Driver and may cause OutOfMemoryError on huge datasets.